# 손실함수 · 옵티마이저 · 활성화함수 교체 실험 + Optuna 베이지안 탐색

기존 `pytorch_loss_functions.ipynb`는 각 손실함수의 **계산 공식**을 이해하는 실습이었습니다.

이 노트북은 한 단계 더 나아가, 실제로 작은 모델을 학습시키면서
**"어떤 손실함수 / 옵티마이저 / 활성화함수 조합이 가장 좋은가"** 를
Optuna 베이지안 탐색(TPE)으로 자동으로 찾아봅니다.

구성:

1. **레지스트리 설정**: 손실/옵티마이저/활성화를 문자열 이름으로 교체 가능하게 만들기
2. **실험 A — 회귀 (이상치 포함 데이터)**: MSE vs MAE vs Huber vs SmoothL1
   - 이상치가 있는 데이터에서는 어떤 손실함수가 유리한지 직접 확인합니다.
3. **실험 B — 다중 분류 (나선형 데이터)**: CrossEntropy vs Label Smoothing vs NLL+LogSoftmax

각 실험에서 Optuna는 손실함수와 함께 옵티마이저, 활성화 함수, 학습률,
은닉층 크기까지 동시에 탐색합니다.

In [ ]:
# ===============================
# 0. Optuna 설치 확인
# ===============================

try:
    import optuna
except ImportError:
    %pip install optuna -q
    import optuna

print("Optuna version:", optuna.__version__)

In [ ]:
# ===============================
# 1. 라이브러리 임포트 및 환경 설정
# ===============================

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)

torch.set_printoptions(precision=4)
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 2. 레지스트리: 구성 요소를 문자열 이름으로 교체하기

`"이름" → 생성 함수` 딕셔너리를 만들어 두면,
설정 딕셔너리의 문자열 하나만 바꿔서 손실함수/옵티마이저/활성화함수를 교체할 수 있습니다.

- 회귀용 손실: 원본 노트북에서 배운 `MSELoss`, `L1Loss`, `HuberLoss` + `SmoothL1Loss`
- 분류용 손실: `CrossEntropyLoss`, label smoothing 변형, `LogSoftmax + NLLLoss`
- Huber의 `delta`처럼 손실함수 자체의 하이퍼파라미터도 키워드 인자로 받아 탐색할 수 있습니다.

In [ ]:
# ===============================
# 2. 레지스트리 정의
# ===============================

# --- 활성화 함수 레지스트리 ---
ACTIVATIONS = {
    "relu":       lambda: nn.ReLU(),
    "leaky_relu": lambda: nn.LeakyReLU(0.01),
    "elu":        lambda: nn.ELU(),
    "gelu":       lambda: nn.GELU(),
    "silu":       lambda: nn.SiLU(),
    "tanh":       lambda: nn.Tanh(),
}

# --- 최적화 알고리즘 레지스트리 ---
OPTIMIZERS = {
    "adam":    lambda params, lr: optim.Adam(params, lr=lr),
    "adamw":   lambda params, lr: optim.AdamW(params, lr=lr, weight_decay=1e-4),
    "rmsprop": lambda params, lr: optim.RMSprop(params, lr=lr),
    "sgd":     lambda params, lr: optim.SGD(params, lr=lr, momentum=0.9),
}

# --- 회귀용 손실 함수 레지스트리 ---
# **kw를 받도록 만들어, huber의 delta처럼 손실함수 고유의
# 하이퍼파라미터도 함께 전달할 수 있게 했습니다.
REG_LOSSES = {
    "mse":       lambda **kw: nn.MSELoss(),
    "mae":       lambda **kw: nn.L1Loss(),
    "huber":     lambda delta=1.0, **kw: nn.HuberLoss(delta=delta),
    "smooth_l1": lambda beta=1.0, **kw: nn.SmoothL1Loss(beta=beta),
}


# NLLLoss는 로그 확률을 입력으로 받으므로 log_softmax를 먼저 적용하는 래퍼입니다.
# 수학적으로 CrossEntropyLoss와 동일하지만, 교체 구조를 보여주기 위해 포함합니다.
class NLLWithLogSoftmax(nn.Module):
    def __init__(self):
        super().__init__()
        self.nll = nn.NLLLoss()

    def forward(self, logits, target):
        return self.nll(F.log_softmax(logits, dim=1), target)


# --- 분류용 손실 함수 레지스트리 ---
CLS_LOSSES = {
    "cross_entropy":  lambda **kw: nn.CrossEntropyLoss(),
    "ce_smooth":      lambda smoothing=0.1, **kw: nn.CrossEntropyLoss(label_smoothing=smoothing),
    "nll_logsoftmax": lambda **kw: NLLWithLogSoftmax(),
}


def make_activation(name):
    return ACTIVATIONS[name]()

def make_optimizer(name, params, lr):
    return OPTIMIZERS[name](params, lr)

def make_reg_loss(name, **kw):
    return REG_LOSSES[name](**kw)

def make_cls_loss(name, **kw):
    return CLS_LOSSES[name](**kw)

print("회귀 손실 후보:", list(REG_LOSSES.keys()))
print("분류 손실 후보:", list(CLS_LOSSES.keys()))
print("옵티마이저 후보:", list(OPTIMIZERS.keys()))
print("활성화 함수 후보:", list(ACTIVATIONS.keys()))

In [ ]:
# ===============================
# 3. 공용 MLP 모델 정의
# ===============================

# 회귀와 분류 양쪽에서 사용할 수 있는 다층 퍼셉트론입니다.
# 은닉층 개수, 은닉 노드 수, 활성화 함수를 모두 인자로 받습니다.
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden_units=64, n_layers=2, activation="relu"):
        super().__init__()
        layers = []
        d = in_dim
        for _ in range(n_layers):
            layers.append(nn.Linear(d, hidden_units))
            layers.append(make_activation(activation))   # 활성화는 레지스트리에서 생성
            d = hidden_units
        layers.append(nn.Linear(d, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# 동작 확인
print(MLP(in_dim=1, out_dim=1, hidden_units=32, n_layers=2, activation="gelu"))

## 실험 A — 회귀: 이상치가 있는 데이터에서 손실함수 비교

`y = 3·sin(x) + 0.5x`에 잡음을 더하고, 전체의 8%에 **큰 이상치**를 섞었습니다.

원본 노트북에서 배운 내용을 떠올려 보세요.

- MSE는 오차를 제곱하므로 이상치에 매우 민감합니다.
- MAE와 Huber는 이상치의 영향을 덜 받습니다.

평가는 어떤 손실로 학습했든 공평하도록 **검증 MAE**로 통일합니다.

In [ ]:
# ===============================
# A-1. 회귀용 합성 데이터 생성
# ===============================

N = 600

x = np.random.uniform(-4, 4, size=(N, 1)).astype(np.float32)
y_clean = (3.0 * np.sin(x) + 0.5 * x).astype(np.float32)

# 기본 잡음을 추가합니다.
y = y_clean + np.random.normal(0, 0.3, size=y_clean.shape).astype(np.float32)

# 전체의 8%를 이상치로 만듭니다. (정답에서 크게 벗어난 값)
n_outlier = int(N * 0.08)
outlier_idx = np.random.choice(N, size=n_outlier, replace=False)
y[outlier_idx] += np.random.normal(0, 8.0, size=(n_outlier, 1)).astype(np.float32)

# 학습/검증 분할 (80% / 20%)
split = int(N * 0.8)
idx = np.random.permutation(N)
train_idx, valid_idx = idx[:split], idx[split:]

x_train = torch.tensor(x[train_idx]).to(device)
y_train = torch.tensor(y[train_idx]).to(device)
x_valid = torch.tensor(x[valid_idx]).to(device)
y_valid = torch.tensor(y[valid_idx]).to(device)

# 데이터 시각화: 이상치가 섞여 있는 모습을 확인합니다.
plt.figure(figsize=(8, 4))
plt.scatter(x, y, s=10, alpha=0.5, label="data (with outliers)")
order = np.argsort(x[:, 0])
plt.plot(x[order], y_clean[order], color="red", linewidth=2, label="true function")
plt.title("Regression Data with Outliers")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ===============================
# A-2. 회귀 실험 함수 정의
# ===============================

REG_EPOCHS = 300   # 데이터가 작으므로 전체 배치(full-batch)로 학습합니다.

def run_regression(config, return_model=False):
    # config 딕셔너리 하나로 모델/손실/옵티마이저를 만들어 학습하고,
    # 학습 중 가장 좋았던 검증 MAE를 반환합니다.
    torch.manual_seed(SEED)   # 조합 비교가 공평하도록 초기 가중치를 고정합니다.

    model = MLP(
        in_dim=1, out_dim=1,
        hidden_units=config["hidden_units"],
        n_layers=config["n_layers"],
        activation=config["activation"],
    ).to(device)

    # 손실함수 고유 파라미터(delta, beta)는 loss_kwargs로 전달합니다.
    criterion = make_reg_loss(config["loss"], **config.get("loss_kwargs", {}))
    optimizer = make_optimizer(config["optimizer"], model.parameters(), config["lr"])

    # 평가 지표는 손실함수와 무관하게 MAE로 통일합니다.
    mae_metric = nn.L1Loss()

    best_mae = float("inf")
    for epoch in range(REG_EPOCHS):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(x_train), y_train)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_mae = mae_metric(model(x_valid), y_valid).item()
        best_mae = min(best_mae, val_mae)

    if return_model:
        return best_mae, model
    return best_mae


# 동작 확인: 기본 조합 하나를 직접 실행해 봅니다.
baseline = {
    "loss": "mse",
    "optimizer": "adam",
    "activation": "relu",
    "lr": 1e-2,
    "hidden_units": 64,
    "n_layers": 2,
}
print("baseline(MSE) 검증 MAE: {:.4f}".format(run_regression(baseline)))

In [ ]:
# ===============================
# A-3. Optuna 목적 함수 (회귀)
# ===============================

def objective_reg(trial):
    loss_name = trial.suggest_categorical("loss", list(REG_LOSSES.keys()))

    # 손실함수에 따라 추가 하이퍼파라미터를 조건부로 탐색합니다.
    loss_kwargs = {}
    if loss_name == "huber":
        loss_kwargs["delta"] = trial.suggest_float("huber_delta", 0.3, 3.0)
    elif loss_name == "smooth_l1":
        loss_kwargs["beta"] = trial.suggest_float("smooth_l1_beta", 0.3, 3.0)

    config = {
        "loss":         loss_name,
        "loss_kwargs":  loss_kwargs,
        "optimizer":    trial.suggest_categorical("optimizer", list(OPTIMIZERS.keys())),
        "activation":   trial.suggest_categorical("activation", ["relu", "leaky_relu", "elu", "gelu", "silu", "tanh"]),
        "lr":           trial.suggest_float("lr", 1e-4, 1e-1, log=True),
        "hidden_units": trial.suggest_categorical("hidden_units", [32, 64, 128]),
        "n_layers":     trial.suggest_int("n_layers", 1, 3),
    }

    return run_regression(config)   # 검증 MAE (작을수록 좋음)


study_reg = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study_reg.optimize(objective_reg, n_trials=30, show_progress_bar=True)

print()
print("=== 회귀 실험 최적 결과 ===")
print("최저 검증 MAE: {:.4f}".format(study_reg.best_value))
for k, v in study_reg.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# ===============================
# A-4. 회귀 결과 분석 및 시각화
# ===============================

# 손실함수별로 가장 좋았던 trial을 비교합니다.
print("손실함수별 최고 성능 (검증 MAE, 작을수록 좋음):")
df_reg = study_reg.trials_dataframe(attrs=("number", "value", "params"))
best_by_loss = df_reg.groupby("params_loss")["value"].min().sort_values()
print(best_by_loss)

# 최적 조합으로 다시 학습하여 예측 곡선을 그립니다.
best_params = study_reg.best_params
best_config = {
    "loss":         best_params["loss"],
    "loss_kwargs":  {},
    "optimizer":    best_params["optimizer"],
    "activation":   best_params["activation"],
    "lr":           best_params["lr"],
    "hidden_units": best_params["hidden_units"],
    "n_layers":     best_params["n_layers"],
}
if best_params["loss"] == "huber":
    best_config["loss_kwargs"]["delta"] = best_params["huber_delta"]
elif best_params["loss"] == "smooth_l1":
    best_config["loss_kwargs"]["beta"] = best_params["smooth_l1_beta"]

best_mae, best_model = run_regression(best_config, return_model=True)

# 예측 곡선 시각화
x_plot = torch.linspace(-4, 4, 200).reshape(-1, 1).to(device)
best_model.eval()
with torch.no_grad():
    y_plot = best_model(x_plot).cpu().numpy()

plt.figure(figsize=(8, 4))
plt.scatter(x, y, s=10, alpha=0.4, label="data (with outliers)")
plt.plot(x[order], y_clean[order], color="red", linewidth=2, label="true function")
plt.plot(x_plot.cpu().numpy(), y_plot, color="green", linewidth=2,
         label=f"best model ({best_config['loss']})")
plt.title(f"Best Regression Model — val MAE {best_mae:.4f}")
plt.legend()
plt.grid(True)
plt.show()

## 실험 B — 다중 분류: 나선형(spiral) 데이터

3개 클래스가 나선 모양으로 얽혀 있는 2차원 데이터입니다.
선형 모델로는 분류할 수 없어서 활성화 함수(비선형성)의 역할이 중요합니다.

탐색 대상:

- 손실 함수: `cross_entropy`, `ce_smooth`(label smoothing 정도도 함께 탐색), `nll_logsoftmax`
- 옵티마이저, 활성화 함수, 학습률, 은닉층 크기/개수

평가는 **검증 정확도**로 통일합니다.

In [ ]:
# ===============================
# B-1. 나선형 분류 데이터 생성
# ===============================

def make_spiral(points_per_class=200, n_classes=3, noise=0.2):
    X = np.zeros((points_per_class * n_classes, 2), dtype=np.float32)
    y = np.zeros(points_per_class * n_classes, dtype=np.int64)
    for c in range(n_classes):
        ix = range(points_per_class * c, points_per_class * (c + 1))
        r = np.linspace(0.0, 1.0, points_per_class)                      # 반지름
        t = np.linspace(c * 4, (c + 1) * 4, points_per_class) \
            + np.random.randn(points_per_class) * noise                  # 각도 + 잡음
        X[ix] = np.c_[r * np.sin(t), r * np.cos(t)]
        y[ix] = c
    return X, y


N_CLASSES = 3
X_sp, y_sp = make_spiral(points_per_class=200, n_classes=N_CLASSES, noise=0.2)

# 학습/검증 분할 (80% / 20%)
n_total = len(X_sp)
idx = np.random.permutation(n_total)
split = int(n_total * 0.8)
train_idx, valid_idx = idx[:split], idx[split:]

Xc_train = torch.tensor(X_sp[train_idx]).to(device)
yc_train = torch.tensor(y_sp[train_idx]).to(device)
Xc_valid = torch.tensor(X_sp[valid_idx]).to(device)
yc_valid = torch.tensor(y_sp[valid_idx]).to(device)

# 데이터 시각화
plt.figure(figsize=(5, 5))
plt.scatter(X_sp[:, 0], X_sp[:, 1], c=y_sp, s=15, cmap="brg")
plt.title("Spiral Classification Data (3 classes)")
plt.grid(True)
plt.show()

In [ ]:
# ===============================
# B-2. 분류 실험 함수 정의
# ===============================

CLS_EPOCHS = 300

def run_classification(config, return_model=False):
    # config 딕셔너리 하나로 모델/손실/옵티마이저를 만들어 학습하고,
    # 학습 중 가장 좋았던 검증 정확도를 반환합니다.
    torch.manual_seed(SEED)

    model = MLP(
        in_dim=2, out_dim=N_CLASSES,
        hidden_units=config["hidden_units"],
        n_layers=config["n_layers"],
        activation=config["activation"],
    ).to(device)

    criterion = make_cls_loss(config["loss"], **config.get("loss_kwargs", {}))
    optimizer = make_optimizer(config["optimizer"], model.parameters(), config["lr"])

    best_acc = 0.0
    for epoch in range(CLS_EPOCHS):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(Xc_train), yc_train)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            pred = model(Xc_valid).argmax(dim=1)
            acc = (pred == yc_valid).float().mean().item()
        best_acc = max(best_acc, acc)

    if return_model:
        return best_acc, model
    return best_acc


# 동작 확인
baseline_cls = {
    "loss": "cross_entropy",
    "optimizer": "adam",
    "activation": "relu",
    "lr": 1e-2,
    "hidden_units": 64,
    "n_layers": 2,
}
print("baseline 검증 정확도: {:.2f}%".format(run_classification(baseline_cls) * 100))

In [ ]:
# ===============================
# B-3. Optuna 목적 함수 (분류)
# ===============================

def objective_cls(trial):
    loss_name = trial.suggest_categorical("loss", list(CLS_LOSSES.keys()))

    loss_kwargs = {}
    if loss_name == "ce_smooth":
        # label smoothing 정도 자체도 탐색합니다.
        loss_kwargs["smoothing"] = trial.suggest_float("smoothing", 0.01, 0.2)

    config = {
        "loss":         loss_name,
        "loss_kwargs":  loss_kwargs,
        "optimizer":    trial.suggest_categorical("optimizer", list(OPTIMIZERS.keys())),
        "activation":   trial.suggest_categorical("activation", ["relu", "leaky_relu", "elu", "gelu", "silu", "tanh"]),
        "lr":           trial.suggest_float("lr", 1e-4, 1e-1, log=True),
        "hidden_units": trial.suggest_categorical("hidden_units", [32, 64, 128]),
        "n_layers":     trial.suggest_int("n_layers", 1, 3),
    }

    return run_classification(config)   # 검증 정확도 (클수록 좋음)


study_cls = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study_cls.optimize(objective_cls, n_trials=30, show_progress_bar=True)

print()
print("=== 분류 실험 최적 결과 ===")
print("최고 검증 정확도: {:.2f}%".format(study_cls.best_value * 100))
for k, v in study_cls.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# ===============================
# B-4. 분류 결과 분석: 결정 경계 시각화
# ===============================

# 손실함수별 최고 성능 비교
df_cls = study_cls.trials_dataframe(attrs=("number", "value", "params"))
print("손실함수별 최고 검증 정확도:")
print(df_cls.groupby("params_loss")["value"].max().sort_values(ascending=False))

# 최적 조합으로 다시 학습합니다.
bp = study_cls.best_params
best_config_cls = {
    "loss":         bp["loss"],
    "loss_kwargs":  ({"smoothing": bp["smoothing"]} if bp["loss"] == "ce_smooth" else {}),
    "optimizer":    bp["optimizer"],
    "activation":   bp["activation"],
    "lr":           bp["lr"],
    "hidden_units": bp["hidden_units"],
    "n_layers":     bp["n_layers"],
}
best_acc, best_model_cls = run_classification(best_config_cls, return_model=True)

# 결정 경계를 그립니다: 2차원 평면을 격자로 채워 각 점의 예측 클래스를 색으로 표시합니다.
xx, yy = np.meshgrid(
    np.linspace(-1.3, 1.3, 300),
    np.linspace(-1.3, 1.3, 300),
)
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()].astype(np.float32)).to(device)

best_model_cls.eval()
with torch.no_grad():
    zz = best_model_cls(grid).argmax(dim=1).cpu().numpy().reshape(xx.shape)

plt.figure(figsize=(6, 6))
plt.contourf(xx, yy, zz, alpha=0.25, cmap="brg")
plt.scatter(X_sp[:, 0], X_sp[:, 1], c=y_sp, s=15, cmap="brg")
plt.title(f"Decision Boundary — val acc {best_acc * 100:.2f}% "
          f"({best_config_cls['loss']}, {best_config_cls['activation']}, {best_config_cls['optimizer']})")
plt.grid(True)
plt.show()

In [ ]:
# ===============================
# 4. 탐색 과정 시각화 (선택)
# ===============================

try:
    from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances

    plot_optimization_history(study_reg)
    plt.title("Regression Study")
    plt.tight_layout()
    plt.show()

    plot_optimization_history(study_cls)
    plt.title("Classification Study")
    plt.tight_layout()
    plt.show()

    plot_param_importances(study_cls)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("시각화 생략:", e)

## 5. 정리: 구성 요소 교체 방법

이 노트북의 모든 실험은 **config 딕셔너리 하나**로 제어됩니다.
Optuna 없이 직접 조합을 실험하려면 아래처럼 하면 됩니다.

```python
my_config = {
    "loss": "huber",                     # REG_LOSSES의 key
    "loss_kwargs": {"delta": 1.5},       # 손실함수 고유 파라미터
    "optimizer": "adamw",                # OPTIMIZERS의 key
    "activation": "gelu",                # ACTIVATIONS의 key
    "lr": 5e-3,
    "hidden_units": 64,
    "n_layers": 2,
}
val_mae = run_regression(my_config)
```

새 구성 요소 추가도 레지스트리에 한 줄이면 됩니다.

```python
ACTIVATIONS["mish"] = lambda: nn.Mish()
OPTIMIZERS["adagrad"] = lambda params, lr: optim.Adagrad(params, lr=lr)
REG_LOSSES["mse_sum"] = lambda **kw: nn.MSELoss(reduction="sum")
CLS_LOSSES["ce_weighted"] = lambda **kw: nn.CrossEntropyLoss(weight=torch.tensor([1.0, 2.0, 1.0]).to(device))
```

핵심 요약:

| 구분 | 원본 노트북 | 이 노트북 |
|---|---|---|
| 손실함수 | 공식 vs PyTorch 계산 비교 | 실제 학습에서 어떤 손실이 유리한지 탐색 |
| 하이퍼파라미터 | 고정 | Optuna TPE 베이지안 탐색 |
| 구성 요소 교체 | 코드 수정 필요 | 문자열(레지스트리 key)만 변경 |